!pip install -U langchain langchain-google-genai langchain-community chromadb faiss-cpu tiktoken wikipedia

In [1]:
from langchain_community.retrievers import WikipediaRetriever

/tmp/ipykernel_7339/2879121110.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.retrievers import WikipediaRetriever


In [2]:
retriever= WikipediaRetriever(top_k_results=2, lang= "en")

In [3]:
query="the geopolitical history of india and pakistan from the perspective of a chinese"

In [4]:
docs= retriever.invoke(query)

In [5]:
docs

[Document(metadata={'title': 'India–Pakistan war of 1971', 'summary': "The India–Pakistan war of 1971, also known as the third Indo-Pakistani war, was a military confrontation between India and Pakistan that occurred during the Bangladesh Liberation War in East Pakistan from 3 December 1971 until the Pakistani capitulation in Dhaka on 16 December 1971.  The war began with Pakistan's Operation Chengiz Khan, consisting of preemptive aerial strikes on eight Indian air stations. The strikes led to India declaring war on Pakistan, marking their entry into the war for East Pakistan's independence, on the side of Bengali nationalist forces. India's entry expanded the existing conflict with Indian and Pakistani forces engaging on both the eastern and western fronts.\nThirteen days after the war started, India achieved a clear upper hand, and the Eastern Command of the Pakistan military signed the instrument of surrender on 16 December 1971 in Dhaka, marking the formation of East Pakistan as th

In [6]:
from langchain_community.vectorstores import Chroma
from langchain_google_genai import GoogleGenerativeAIEmbeddings
from langchain_core.documents import Document

In [7]:
# Step 1: Your source documents
documents = [
    Document(page_content="LangChain helps developers build LLM applications easily."),
    Document(page_content="Chroma is a vector database optimized for LLM-based search."),
    Document(page_content="Embeddings convert text into high-dimensional vectors."),
    Document(page_content="OpenAI provides powerful embedding models."),
]

In [45]:
from dotenv import load_dotenv
import os

load_dotenv()


True

In [9]:
embedding_model= GoogleGenerativeAIEmbeddings(
    model="models/gemini-embedding-001"
)
    

In [10]:
vectorstore= Chroma.from_documents(
    documents= documents,
    embedding= embedding_model,
    collection_name="My_collection"
    
)

In [11]:
retriever= vectorstore.as_retriever(search_kwargs={"k":2})

In [12]:
query="what is chroma used for" 

In [13]:
results=retriever.invoke(query)

In [14]:
for i, doc in enumerate(results):
    print(f"\n--- Result {i+1} ---")
    print(doc.page_content)


--- Result 1 ---
Chroma is a vector database optimized for LLM-based search.

--- Result 2 ---
Embeddings convert text into high-dimensional vectors.


In [15]:
results = vectorstore.similarity_search(query, k=2)

In [16]:
for i, doc in enumerate(results):
    print(f"\n--- Result {i+1} ---")
    print(doc.page_content)


--- Result 1 ---
Chroma is a vector database optimized for LLM-based search.

--- Result 2 ---
Embeddings convert text into high-dimensional vectors.


MMR


In [17]:
from langchain_community.vectorstores import FAISS
from langchain_google_genai import GoogleGenerativeAIEmbeddings
from langchain_core.documents import Document

In [18]:
# Sample documents
docs = [
    Document(page_content="LangChain makes it easy to work with LLMs."),
    Document(page_content="LangChain is used to build LLM based applications."),
    Document(page_content="Chroma is used to store and search document embeddings."),
    Document(page_content="Embeddings are vector representations of text."),
    Document(page_content="MMR helps you get diverse results when doing similarity search."),
    Document(page_content="LangChain supports Chroma, FAISS, Pinecone, and more."),
]

In [44]:
from dotenv import load_dotenv
import os

load_dotenv()


True

In [20]:
embedding_model= GoogleGenerativeAIEmbeddings(
    model="models/gemini-embedding-001"
)
    

In [21]:
from langchain_community.vectorstores import FAISS

vector_store= FAISS.from_documents(

    embedding= embedding_model,
    documents= docs,
   
)

In [22]:
query= "what is langchain"


In [23]:
retriever= vector_store.as_retriever(
    search_type="mmr",
    search_kwargs={
        "k":3,
        "lambda_mult":0
    }
    
)

In [24]:
results= retriever.invoke(query)

In [25]:
results

[Document(id='9676ceb7-1570-486d-b211-53554731f702', metadata={}, page_content='LangChain is used to build LLM based applications.'),
 Document(id='2ae65d05-d80f-4424-b7ac-0a2deb904496', metadata={}, page_content='MMR helps you get diverse results when doing similarity search.'),
 Document(id='a6e544e6-0750-48d7-831a-d5b03092c73f', metadata={}, page_content='Chroma is used to store and search document embeddings.')]

In [26]:
from langchain_classic.retrievers.multi_query import MultiQueryRetriever

In [27]:
# Relevant health & wellness documents
all_docs = [
    Document(page_content="Regular walking boosts heart health and can reduce symptoms of depression.", metadata={"source": "H1"}),
    Document(page_content="Consuming leafy greens and fruits helps detox the body and improve longevity.", metadata={"source": "H2"}),
    Document(page_content="Deep sleep is crucial for cellular repair and emotional regulation.", metadata={"source": "H3"}),
    Document(page_content="Mindfulness and controlled breathing lower cortisol and improve mental clarity.", metadata={"source": "H4"}),
    Document(page_content="Drinking sufficient water throughout the day helps maintain metabolism and energy.", metadata={"source": "H5"}),
    Document(page_content="The solar energy system in modern homes helps balance electricity demand.", metadata={"source": "I1"}),
    Document(page_content="Python balances readability with power, making it a popular system design language.", metadata={"source": "I2"}),
    Document(page_content="Photosynthesis enables plants to produce energy by converting sunlight.", metadata={"source": "I3"}),
    Document(page_content="The 2022 FIFA World Cup was held in Qatar and drew global energy and excitement.", metadata={"source": "I4"}),
    Document(page_content="Black holes bend spacetime and store immense gravitational energy.", metadata={"source": "I5"}),
]

In [28]:
vector_store= FAISS.from_documents(documents= all_docs, embedding=embedding_model)

In [29]:
similarity_retriever= vector_store. as_retriever(search_type="similarity", search_kwargs={'k': 5})

In [30]:
from langchain_google_genai import ChatGoogleGenerativeAI

llm = ChatGoogleGenerativeAI(
    model="gemini-3.5-flash",
    temperature=0.7
)

In [31]:
multiquery_retriever= MultiQueryRetriever.from_llm(

    retriever= vector_store.as_retriever(search_kwargs={'k':5}),
    llm= llm)

In [32]:
# Query
query = "How to improve energy levels and maintain balance?"

In [33]:
results1= similarity_retriever.invoke(query)

In [34]:
results2= multiquery_retriever.invoke(query)

In [35]:
from langchain_classic.retrievers import ContextualCompressionRetriever
from langchain_classic.retrievers.document_compressors import LLMChainExtractor

In [36]:
# Recreate the document objects from the previous data
docs = [
    Document(page_content=(
        """The Grand Canyon is one of the most visited natural wonders in the world.
        Photosynthesis is the process by which green plants convert sunlight into energy.
        Millions of tourists travel to see it every year. The rocks date back millions of years."""
    ), metadata={"source": "Doc1"}),

    Document(page_content=(
        """In medieval Europe, castles were built primarily for defense.
        The chlorophyll in plant cells captures sunlight during photosynthesis.
        Knights wore armor made of metal. Siege weapons were often used to breach castle walls."""
    ), metadata={"source": "Doc2"}),

    Document(page_content=(
        """Basketball was invented by Dr. James Naismith in the late 19th century.
        It was originally played with a soccer ball and peach baskets. NBA is now a global league."""
    ), metadata={"source": "Doc3"}),

    Document(page_content=(
        """The history of cinema began in the late 1800s. Silent films were the earliest form.
        Thomas Edison was among the pioneers. Photosynthesis does not occur in animal cells.
        Modern filmmaking involves complex CGI and sound design."""
    ), metadata={"source": "Doc4"})
]

In [37]:
# Create a FAISS vector store from the docume
vectorstore = FAISS.from_documents(docs, embedding_model)

In [38]:
base_retriever = vectorstore.as_retriever(search_kwargs={"k": 5})

In [39]:
compressor= LLMChainExtractor.from_llm(llm)

In [40]:
compression_retriever= ContextualCompressionRetriever(
    base_retriever= base_retriever,
    base_compressor= compressor)

In [41]:
query= "what is photosynthesis"

In [42]:
results=compression_retriever.invoke(query)

In [43]:
results

[Document(metadata={'source': 'Doc1'}, page_content='Photosynthesis is the process by which green plants convert sunlight into energy.'),
 Document(metadata={'source': 'Doc2'}, page_content='The chlorophyll in plant cells captures sunlight during photosynthesis.')]